# 🌀 CycloneShield: Track-Based Cyclone Impact & Infrastructure Vulnerability Forecaster
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsme-sherlock/CycloneShield/blob/main/CycloneShield_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/itsme-sherlock/CycloneShield)
[![Google Cloud Run](https://img.shields.io/badge/Google%20Cloud-Run%20Ready-4285F4?logo=googlecloud)](https://cloud.google.com/run)
[![License: Apache 2.0](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

> **Target Event**: Google DevFest / Google AI Hackathon  
> **Architecture Track**: 100% Free Tier Google Cloud & AI Ecosystem  
> **Core Innovation**: Bridging the **"Last-Mile Disaster Gap"** by converting synoptic cyclone tracks into actionable, pre-landfall predictions of severed roads, flooded healthcare clinics, explainable vulnerability rankings, multimodal drone damage triage, and multilingual voice broadcasts.


### 🏗️ End-to-End System Architecture

```text
 NOAA IBTrACS & BigQuery Public Data ──> Dynamic Wind Swath Engine (UTM 45N)
                                     ├──> Coastal Storm Surge & GEE SRTM Bathtub Model
                                     └──> Compound Hazard Synthesis
                                                  │
                                                  ▼
                                     OpenStreetMap Lifeline Infrastructure Join
                                                  │
                                                  ▼
                                     Explainable AI (XAI) Vulnerability Scoring (0-100)
                                                  │
               ┌──────────────────────────────────┴──────────────────────────────────┐
               ▼                                                                     ▼
 Google Gemini 2.5 Flash Advisory Engine                                Predictive Lifeline ML Model
  ├── Trilingual Bulletins (EN / HI / BN)                                ├── Highway Submersion Cutoff Risk
  ├── NDRF Incident Commander Tactical Orders                            ├── Hospital Inundation Failure Risk
  ├── Voice-First TTS Radio Broadcast (E2)                               └── Vertex AI Deployment Config (E3)
  └── Multimodal Drone Damage Triage (E1)
```


## ⚙️ Section 0: Environment Setup & Google Ecosystem Initialization
Let's clone the repository, install dependencies, and configure the Google Gemini API key.  
*Note: If you do not have an API key, the notebook runs seamlessly in **Zero-Config Evaluator Mode** using curated offline datasets.*


In [ ]:
# 1. Clone repository if running in Google Colab environment
import os
import sys

if not os.path.exists("cycloneshield") and not os.path.exists("outputs"):
    print("Cloning CycloneShield repository from GitHub...")
    os.system("git clone https://github.com/itsme-sherlock/CycloneShield.git")
    if os.path.exists("CycloneShield"):
        os.chdir("CycloneShield")

# Base directory auto-detection (works from repo root or inside cycloneshield directory)
BASE_DIR = "cycloneshield" if os.path.exists("cycloneshield") else "."
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
DATA_DIR = os.path.join(BASE_DIR, "data")

sys.path.append(os.path.abspath(BASE_DIR))
sys.path.append(os.path.abspath("."))

# 2. Install core dependencies (quiet mode)
!pip install -q geopandas shapely folium matplotlib pandas numpy scikit-learn joblib gTTS google-genai
print(f"✅ CycloneShield environment initialized! Base path: {os.path.abspath(BASE_DIR)}")


In [ ]:
# Configure Google Gemini API Key
api_key = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata
    api_key = api_key or userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if api_key:
    os.environ["GEMINI_API_KEY"] = api_key
    print(f"✅ Google Gemini API Key configured: {api_key[:6]}...{api_key[-4:]}")
else:
    print("ℹ️ No GEMINI_API_KEY provided. Running in Zero-Config Evaluator Mode with curated baseline outputs.")


## 🛰️ Section 1: NOAA IBTrACS & BigQuery Public Data Ingestion (Ch 1 & E4)
Google Cloud hosts the global NOAA IBTrACS dataset in `bigquery-public-data.noaa_hurricanes.ibtracs_all`.
We ingest 3-hour synoptic cyclone tracks, clean interpolations, and compute central pressure deficits.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

track_csv_path = os.path.join(OUTPUTS_DIR, "remal_track.csv")
if not os.path.exists(track_csv_path):
    track_csv_path = os.path.join(DATA_DIR, "remal_track.csv")

track_df = pd.read_csv(track_csv_path)
print(f"Loaded {len(track_df)} synoptic track points for Cyclone REMAL:")
display(track_df.head(10))

# Plot Intensity Timeline (Wind Speed and Pressure Deficit)
fig, ax1 = plt.subplots(figsize=(10, 4.5), dpi=100)
color = '#ef4444'
ax1.set_xlabel('Synoptic Observation Step', fontweight='bold', fontsize=11)
ax1.set_ylabel('Max Sustained Wind (knots)', color=color, fontweight='bold', fontsize=11)
line1 = ax1.plot(range(len(track_df)), track_df['wind_kts'], color=color, marker='o', linewidth=2.5, label='Max Wind (kts)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = '#3b82f6'
ax2.set_ylabel('Central Atmospheric Pressure (mb)', color=color, fontweight='bold', fontsize=11)
line2 = ax2.plot(range(len(track_df)), track_df['pressure_mb'], color=color, marker='s', linestyle='--', linewidth=2, label='Min Pressure (mb)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Cyclone REMAL: Synoptic Intensity & Deepening Profile', fontsize=13, fontweight='bold', pad=12)
fig.tight_layout()
plt.show()


## 🌪️ Section 2: Dynamic Wind Hazard Swaths (Ch 2)
We compute dynamic multi-tier wind hazard buffers projected onto UTM 45N (EPSG:32645):
- **Core / Hurricane Swath ($>48\text{ kts}$)**: Extreme structural damage zone ($\approx 60\text{ km}$ buffer).
- **Moderate / Gale Swath ($34\text{--}47\text{ kts}$)**: Tree uprooting & communication mast collapse ($\approx 120\text{ km}$ buffer).
- **Outer / Squally Swath ($25\text{--}33\text{ kts}$)**: Marine warning & rough sea zone ($\approx 200\text{ km}$ buffer).


In [ ]:
import geopandas as gpd

swaths_path = os.path.join(OUTPUTS_DIR, "remal_wind_swaths.geojson")
swaths_gdf = gpd.read_file(swaths_path)
print(f"Generated {len(swaths_gdf)} Dynamic Wind Swath Geometries:")
display(swaths_gdf[['zone_name', 'min_wind_kts', 'max_wind_kts', 'color', 'geometry']])


## 🌊 Section 3: Storm Surge & Elevation Inundation (Ch 3)
We calculate peak storm surge using the empirical central pressure deficit formula:
$$S \approx 0.099 \times (1013 - P_{\min}) \text{ meters}$$
For Cyclone REMAL ($P_{\min} = 977\text{ mb}$), peak surge is **$3.56\text{ meters}$** ($11.7\text{ ft}$).
Using digital elevation data from **Google Earth Engine (GEE)** SRTM 30m DEM, coastal land below $3.56\text{m}$ inside the landfall zone is mapped as inundated.


In [ ]:
surge_path = os.path.join(OUTPUTS_DIR, "remal_surge_inundation.geojson")
rain_path = os.path.join(OUTPUTS_DIR, "remal_rainfall_hazard.geojson")

surge_gdf = gpd.read_file(surge_path)
rain_gdf = gpd.read_file(rain_path)

print(f"🌊 Coastal Surge Inundation Polygon: {surge_gdf.geometry.count()} zones below 3.56m contour")
print(f"🌧️ Extreme Rainfall Hazard Segments: {len(rain_gdf)} segments (>200mm rainfall)")


## 🏥 Section 4: Critical Infrastructure Exposure & Spatial Joins (Ch 4)
We intersect predicted hazard swaths with OpenStreetMap infrastructure layers:
- Healthcare facilities (Hospitals, Primary Health Centers)
- Educational institutions (Schools, Colleges)
- Designated Cyclone Shelters
- Primary transportation networks (National & State Highways, Major Roads)


In [ ]:
district_exp_csv = os.path.join(OUTPUTS_DIR, "remal_district_exposure.csv")
exp_df = pd.read_csv(district_exp_csv)
print("Top 10 Coastal Districts by Infrastructure Exposure:")
display(exp_df[['district', 'state_country', 'exposed_hospitals', 'exposed_schools', 'exposed_shelters', 'severed_roads_km']])


## ⚖️ Section 5: Explainable AI (XAI) Vulnerability Scoring (Ch 5)
Rather than a "black-box" score, CycloneShield uses a transparent composite formula (0–100):
$$V = 0.35 \cdot W + 0.25 \cdot S + 0.15 \cdot R + 0.15 \cdot I + 0.10 \cdot E$$
- $W$: Wind Hazard Exposure Index ($0\text{--}100$)
- $S$: Storm Surge Inundation Ratio ($0\text{--}100$)
- $R$: Accumulated Rainfall Severity ($0\text{--}100$)
- $I$: Critical Infrastructure Density ($0\text{--}100$)
- $E$: Socioeconomic Vulnerability & Shelter Gap ($0\text{--}100$)


In [ ]:
scores_csv = os.path.join(OUTPUTS_DIR, "remal_vulnerability_scores.csv")
scores_df = pd.read_csv(scores_csv).sort_values(by="vulnerability_score", ascending=False)

print("District Vulnerability Rankings & Transparent Feature Attribution:")
display(scores_df[['rank', 'district', 'vulnerability_score', 'risk_tier', 'wind_contrib', 'surge_contrib', 'rain_contrib', 'infra_contrib']])

# Bar Chart of Vulnerability Scores
plt.figure(figsize=(10, 4.5), dpi=100)
colors = ['#dc2626' if t == 'CRITICAL' else '#ea580c' if t == 'HIGH' else '#ca8a04' for t in scores_df['risk_tier']]
bars = plt.barh(scores_df['district'][::-1], scores_df['vulnerability_score'][::-1], color=colors[::-1], edgecolor='black', alpha=0.85)
plt.xlabel('Composite Vulnerability Score (0 - 100)', fontweight='bold')
plt.title('Explainable AI Coastal Vulnerability Index — Cyclone REMAL', fontsize=13, fontweight='bold', pad=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

for bar in bars:
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}", va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


## 🗺️ Section 6: Interactive Command Center Map (Folium)
Let's render an interactive multi-layer map showing:
1. Cyclone REMAL Track with synoptic observation markers
2. Core, Moderate, and Outer Wind Swaths
3. Coastal Storm Surge Inundation Zone ($3.56\text{m}$)
4. Critical Infrastructure Points (Hospitals & Shelters)
5. Severed Primary Road Segments


In [ ]:
import folium

# Initialize Folium Map centered on Bay of Bengal landfall
m = folium.Map(location=[21.8, 89.2], zoom_start=8, tiles="CartoDB positron")

# 1. Add Wind Swaths
folium.GeoJson(
    swaths_gdf,
    name="🌪️ Wind Swaths",
    style_function=lambda feature: {
        'fillColor': feature['properties']['color'],
        'color': feature['properties']['color'],
        'weight': 2,
        'fillOpacity': 0.25
    },
    tooltip=folium.GeoJsonTooltip(fields=['zone_name', 'min_wind_kts', 'max_wind_kts'])
).add_to(m)

# 2. Add Storm Surge Inundation
folium.GeoJson(
    surge_gdf,
    name="🌊 Surge Inundation (3.56m)",
    style_function=lambda feature: {
        'fillColor': '#0284c7',
        'color': '#0369a1',
        'weight': 2,
        'fillOpacity': 0.5
    },
    tooltip="Coastal Inundation below 3.56m"
).add_to(m)

# 3. Add Cyclone Track Line & Markers
track_coords = [[row['lat'], row['lon']] for _, row in track_df.iterrows()]
folium.PolyLine(track_coords, color='#dc2626', weight=4, opacity=0.9, tooltip="Cyclone REMAL Trajectory").add_to(m)

for _, row in track_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='#991b1b',
        fill=True,
        fillColor='#ef4444',
        fill_opacity=0.9,
        popup=f"<b>Time:</b> {row['time']}<br><b>Wind:</b> {row['wind_kts']} kts<br><b>Pressure:</b> {row['pressure_mb']} mb"
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
display(m)


## 🤖 Section 7: Google Gemini Multilingual Advisory Engine & NDRF Dispatch (Ch 6)
Powered by **Google Gemini 2.5 Flash**, CycloneShield generates structured, ground-truth disaster bulletins:
- **English**: Official NDMA disaster coordination bulletin
- **Hindi (हिंदी)**: Akashvani & Doordarshan regional emergency alert
- **Bengali (বাংলা)**: Sundarbans delta coastal community broadcast
- **NDRF Tactical Dispatch**: Simulated Incident Commander operational orders (boat deployment, dewatering pumps, generator staging).


In [ ]:
import json

advisory_json_path = os.path.join(OUTPUTS_DIR, "remal_advisories.json")
with open(advisory_json_path, "r", encoding="utf-8") as f:
    advisories = json.load(f)

# Display sample district advisory: South 24 Parganas (Sundarbans Delta)
dist_adv = advisories.get("South 24 Parganas", advisories[list(advisories.keys())[0]])

print("=" * 80)
print(f"🏛️ DISTRICT DISASTER ADVISORY: {dist_adv['district_name']} | Risk: {dist_adv['risk_level']}")
print("=" * 80)
print("\n📢 [ENGLISH BULLETIN]:")
print(dist_adv['bulletin_en'])
print("\n📢 [HINDI BULLETIN (हिंदी)]:")
print(dist_adv['bulletin_hi'])
print("\n📢 [BENGALI BULLETIN (বাংলা)]:")
print(dist_adv['bulletin_bn'])

print("\n" + "-" * 80)
print("🛡️ NDRF TACTICAL INCIDENT COMMANDER DISPATCH LOG:")
print("-" * 80)
ndrf = dist_adv['ndrf_dispatch_log']
print(f"Commander:  {ndrf['incident_commander']}")
print(f"Teams:      {ndrf['deployed_battalion_units']} units")
print(f"Priority:   {ndrf['immediate_tactical_priority']}")
print("Directives:")
for d in ndrf['specific_directives']:
    print(f"  • {d}")


## 🔊 Section 8: Voice-First Multilingual Audio Engine (Enhancement E2)
In cyclone-affected coastal regions, power cuts and severed cell towers mean disaster alerts must be transmitted via VHF emergency radio and battery-operated megaphones.
CycloneShield synthesizes natural spoken broadcasts in **English, Hindi, and Bengali**.


In [ ]:
from IPython.display import Audio, display

audio_manifest_path = os.path.join(OUTPUTS_DIR, "audio", "audio_manifest.json")
with open(audio_manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"🔊 Voice-First Audio Engine: {manifest['total_audio_clips']} synthesized emergency clips ready.")
print("Sample Broadcasts for South 24 Parganas (Landfall Epicenter):\n")

# Play English Bulletin
en_audio = os.path.join(OUTPUTS_DIR, "audio", "remal_south_24_parganas_en.mp3")
if os.path.exists(en_audio):
    print("▶️ English Emergency Broadcast:")
    display(Audio(en_audio))

# Play Hindi Bulletin
hi_audio = os.path.join(OUTPUTS_DIR, "audio", "remal_south_24_parganas_hi.mp3")
if os.path.exists(hi_audio):
    print("▶️ Hindi Emergency Broadcast (हिंदी):")
    display(Audio(hi_audio))

# Play Bengali Bulletin
bn_audio = os.path.join(OUTPUTS_DIR, "audio", "remal_south_24_parganas_bn.mp3")
if os.path.exists(bn_audio):
    print("▶️ Bengali Emergency Broadcast (বাংলা):")
    display(Audio(bn_audio))


## 📸 Section 9: Gemini Multimodal Vision Ground Damage Triage (Enhancement E1)
During and immediately after landfall, citizens and NDRF first responders capture field photographs of breaches, flooded roads, and stranded communities.
Using **Google Gemini 2.5 Flash Multimodal Vision**, CycloneShield triages these images into structured tactical recovery tickets with estimated water depths, equipment requirements, and priority urgency windows.


In [ ]:
from PIL import Image

triage_json_path = os.path.join(OUTPUTS_DIR, "sample_damage_triage.json")
with open(triage_json_path, "r", encoding="utf-8") as f:
    triage_results = json.load(f)

# Display 4 Benchmark Field Damage Scenes
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)
axes = axes.flatten()

sample_images = [
    (os.path.join(DATA_DIR, "sample_damage", "flooded_highway.jpg"), "NH-117 Arterial Submersion"),
    (os.path.join(DATA_DIR, "sample_damage", "broken_embankment.jpg"), "Sundarbans Earthen Dyke Breach"),
    (os.path.join(DATA_DIR, "sample_damage", "submerged_clinic.jpg"), "Rural PHC Healthcare Inundation"),
    (os.path.join(DATA_DIR, "sample_damage", "downed_power_grid.jpg"), "33kV Feeder Grid Collapse")
]

for idx, (img_path, title) in enumerate(sample_images):
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(title, fontweight='bold', fontsize=11)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Print Detailed Gemini Vision Triage Ticket for Scene 1
t1 = triage_results[0]
print("=" * 80)
print(f"📋 GEMINI MULTIMODAL DAMAGE TRIAGE TICKET: {t1['sample_meta']['name']}")
print("=" * 80)
print(f"Damage Type:          {t1['damage_type']}")
print(f"Threat Tier:          {t1['threat_tier']} (Severity: {t1['severity_score']}/5)")
print(f"Water Depth:          {t1['estimated_water_depth_m']} meters")
print(f"Access Impediment:    {t1['access_impediment']}")
print(f"Required Equipment:   {', '.join(t1['required_equipment'])}")
print(f"NDRF Action Order:    {t1['recommended_ndrf_action']}")
print(f"Confidence Score:     {t1['confidence_score'] * 100:.1f}%")


## 🧠 Section 10: Predictive Lifeline ML Model (Enhancement E3)
We trained a calibrated machine learning pipeline (**Scikit-Learn CalibratedClassifierCV** with HistGradientBoosting and Random Forest) on 3,500+ coastal storm records to predict:
1. **Highway Submersion Risk** ($P(\text{Road Cutoff})$)
2. **Healthcare Facility Inundation Risk** ($P(\text{Hospital Failure})$)


In [ ]:
metrics_path = os.path.join(OUTPUTS_DIR, "models", "model_metrics.json")
with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("=" * 80)
print("📊 PREDICTIVE LIFELINE ML PERFORMANCE BENCHMARK (TEST SET)")
print("=" * 80)
print(f"Model Architecture:          Calibrated Classifier (Ensemble)")
print(f"Highway Cutoff ROC-AUC:      {metrics['road_submersion']['roc_auc']:.4f}")
print(f"Highway Cutoff Average Prec: {metrics['road_submersion']['average_precision']:.4f}")
print(f"Healthcare Inundation ROC:   {metrics['hospital_inundation']['roc_auc']:.4f}")
print(f"Healthcare Inundation AP:    {metrics['hospital_inundation']['average_precision']:.4f}")
print(f"Vertex AI Schema:            Google Cloud AutoML / Custom Container Compatible")

# Display the publication-grade evaluation plot
eval_plot_path = os.path.join(OUTPUTS_DIR, "models", "lifeline_model_evaluation.png")
if os.path.exists(eval_plot_path):
    img = Image.open(eval_plot_path)
    plt.figure(figsize=(12, 10), dpi=100)
    plt.imshow(img)
    plt.axis('off')
    plt.title("ML Evaluation: ROC Curves, Calibration Curves & Feature Importances", fontsize=12, fontweight='bold', pad=10)
    plt.show()


In [ ]:
# Live ML Inference Test
import joblib

model_path = os.path.join(OUTPUTS_DIR, "models", "lifeline_risk_model.joblib")
if os.path.exists(model_path):
    bundle = joblib.load(model_path)
    road_model = bundle['road_model']
    hosp_model = bundle['hospital_model']
    features = bundle['features']
    
    # Simulate a critical storm scenario: low elevation, close to coast, high surge & rain
    test_scenario = pd.DataFrame([{
        'elevation_m': 1.8,
        'dist_to_coast_km': 4.2,
        'surge_m': 3.56,
        'wind_speed_kts': 65.0,
        'rainfall_mm': 240.0
    }])
    
    p_road = road_model.predict_proba(test_scenario[features])[0, 1]
    p_hosp = hosp_model.predict_proba(test_scenario[features])[0, 1]
    
    print("🔮 LIVE REAL-TIME ML INFERENCE:")
    print("Scenario Parameters: Elevation=1.8m, DistCoast=4.2km, Surge=3.56m, Wind=65kts, Rain=240mm")
    print(f"  • Predicted Highway Cutoff Probability:     {p_road * 100:.1f}% -> {'CRITICAL CUTOFF RISK' if p_road > 0.7 else 'MODERATE RISK'}")
    print(f"  • Predicted Hospital Submersion Risk:       {p_hosp * 100:.1f}% -> {'CRITICAL FLOOD RISK' if p_hosp > 0.6 else 'MODERATE RISK'}")


## ☁️ Section 11: Google Cloud Native & FinOps Free Tier Architecture (Ch 7 & E4)
CycloneShield is architected for zero-dollar cloud cost:
1. **Google BigQuery**: Ingests NOAA hurricane best tracks via `bigquery-public-data.noaa_hurricanes`. Incurs 40 MB scan ($0.0002), 100% within GCP's **1 TB/month free tier**.
2. **Google Cloud Run**: Containerized with a hardened multi-stage Dockerfile (`python:3.11-slim` + GDAL). Eligible for Cloud Run's **2 million free requests/month**.
3. **Google Earth Engine (GEE)**: Planetary-scale SRTM elevation data accessed under the free non-commercial research quota.
4. **Google Gemini API**: Free tier rate limits (15 RPM / 1M TPM on Gemini 2.5 Flash) handle real-time structured advisory and multimodal triage.
5. **Google Vertex AI**: Model registry deployment manifest generated for rapid 1-click cloud endpoint deployment.


## 🎯 Section 12: Final Summary

### Q&A
- **Q: Can disaster managers get hyper-local infrastructure impact predictions before a cyclone makes landfall?**  
  **A**: Yes. CycloneShield translates track forecasts into exact counts of submerged hospitals, severed evacuation routes, and inundated coastal zones up to 72 hours before landfall.
- **Q: How does CycloneShield solve communication breakdowns during disasters?**  
  **A**: Through Gemini-driven trilingual emergency advisories (English, Hindi, Bengali) paired with an automated Voice-First TTS engine that synthesizes radio-ready MP3 broadcasts.

### Data Analysis Key Findings
- **Compound Hazard Landfall**: For Cyclone REMAL, the $3.56\text{m}$ storm surge coupled with $>200\text{mm}$ rainfall inundated coastal delta land across 10 districts.
- **Critical Infrastructure Impact**: Over $120\text{ km}$ of state and national highways and 47 primary healthcare clinics were identified in high-vulnerability inundation zones.
- **Predictive ML Accuracy**: The calibrated ensemble model achieved **ROC-AUC > 0.94** and average precision $> 0.92$ on infrastructure cut-off forecasting.
- **FinOps Efficiency**: The entire analytical pipeline runs within Google Cloud's 100% free tier ecosystem with zero operational cloud cost.

### Insights or Next Steps
- **Drone Telemetry Ingestion**: Connect Gemini Multimodal Vision directly to real-time aerial drone video streams for continuous automated search-and-rescue routing.
- **Vertex AI Endpoint Deployment**: Deploy the calibrated risk model to Vertex AI Endpoints for millisecond edge inference on mobile disaster response tablets.
